In [ ]:
# ==========================================
# CELL 1: Environment Setup & System Imports
# ==========================================

# 1. Install required packages
!pip install --force-reinstall "numpy==1.26.4" "pandas==2.0.3" --break-system-packages
!pip install yfinance Pillow matplotlib imageio torch torchvision --break-system-packages

# 2. System and Environment utilities
import os
import sys
import time

In [ ]:
# ==============================================================================
# CELL 2: Project Architecture Imports
# ==============================================================================
import os

# Data Analysis Stack
import numpy as np
import pandas as pd
import yfinance as yf

# Deep Learning Framework
import torch
import torchvision
from torch.utils.data import TensorDataset, DataLoader

# Computer Vision & Image Processing
import imageio
from PIL import Image, ImageDraw, ImageFont

# Data Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 1. Data Preprocessing

## 1. Upload data

In [ ]:
!nvidia-smi

In [ ]:
# 1. Fetch the data
gold = yf.Ticker('GLD').history(start='2005-01-01', end='2025-12-31')
gold = gold[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Keep a snapshot of the original data so we can see what changes
original_gold = gold.copy()

# 2. Recorrect High and Low 
gold['High'] = gold[['High', 'Open', 'Close']].max(axis=1)
gold['Low'] = gold[['Low', 'Open', 'Close']].min(axis=1)

# 3. Track the changes
# Find rows where the new High is different from the old High
fixed_highs = gold[gold['High'] != original_gold['High']]
# Find rows where the new Low is different from the old Low
fixed_lows = gold[gold['Low'] != original_gold['Low']]

print(f"Total Highs corrected: {len(fixed_highs)}")
print(f"Total Lows corrected: {len(fixed_lows)}\n")

if not fixed_highs.empty or not fixed_lows.empty:
    print("Example of a corrected row (Original vs Corrected):")
    # Let's grab the first date that got fixed
    first_fixed_date = (pd.concat([fixed_highs, fixed_lows])).index[0]
    
    print("\n--- ORIGINAL ---")
    print(original_gold.loc[[first_fixed_date], ['Open', 'High', 'Low', 'Close']])
    print("\n--- CORRECTED ---")
    print(gold.loc[[first_fixed_date], ['Open', 'High', 'Low', 'Close']])
    print("\n")

# 4. View the final corrected data
print(gold)

## 2. Preprocessing

In [ ]:
gold.info()
gold.describe()

In [ ]:
gold['Volume'] = gold['Volume'].astype(float)

In [ ]:
gold.info()
gold.describe()

In [ ]:
# Convert index to datetime (if it's not already)
gold.index = pd.to_datetime(gold.index)
# Then sort the DataFrame by date (earliest to latest)
gold = gold.sort_index(ascending=True)

In [ ]:
gold

In [ ]:
gold.isna().sum()

In [ ]:
# Compute 20-day Moving Average on the Close price
gold['MA20'] = gold['Close'].rolling(window=20).mean()
gold = gold.dropna(subset=['MA20'])

In [ ]:
passes = []

# Train is strictly a rolling 10-year window
train_starts = [2005, 2005, 2005, 2005, 2005, 2005, 2005, 2005, 2005, 2005]
train_ends   = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
# 1-year validation immediately after trainin
val_starts   = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
val_ends     = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
# 1-year testing immediately after validatio
test_starts  = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
test_ends    = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

def slice_period(df, start_year, end_year):
    return df[(df.index.year >= start_year) & (df.index.year <= end_year)]

for i in range(len(train_starts)):
    # 1. Base Slices
    train_df = slice_period(gold, train_starts[i], train_ends[i])
    val_df   = slice_period(gold, val_starts[i],   val_ends[i])
    test_df_original = slice_period(gold, test_starts[i],  test_ends[i])
    
    # 2. STITCH ONLY VAL -> TEST
    lookback = 9  
    original_test_len = len(test_df_original)
    
    # Grab the last 4 days of Validation and attach to the front of Testing
    if len(val_df) >= lookback:
        test_df = pd.concat([val_df.iloc[-lookback:], test_df_original])
    else:
        test_df = test_df_original.copy()
    
    # 3. Save to passes
    passes.append({
        "pass": i+1,
        "train_years": (train_starts[i], train_ends[i]),
        "val_years":   (val_starts[i],   val_ends[i]),
        "test_years":  (test_starts[i],  test_ends[i]),
        "train_df": train_df,
        "val_df":   val_df,
        "test_df":  test_df,
        "test_orig_len": original_test_len # Storing this for the clear printout
    })

# --- VERIFICATION PRINT ---
for p in passes:
    lookback_added = len(p['test_df']) - p['test_orig_len']
    
    print(
        f"Pass {p['pass']:>2}: "
        f"Train {p['train_years'][0]}–{p['train_years'][1]}, "
        f"Val {p['val_years'][0]}–{p['val_years'][1]}, "
        f"Test {p['test_years'][0]}–{p['test_years'][1]}, "
        f"Sizes: Train={len(p['train_df'])}, Val={len(p['val_df'])}, "
        f"Test={p['test_orig_len']} + {lookback_added}"
    )

## 2. Simple Return for Normalize Image

### Snippet 1: Momentum (Past)
This snippet creates an **input feature** representing past market movement.

* **Concept:** Past Return (Input Feature)
* **Direction:** Backward-looking
* **Window:** 4 Days
* **Formula:** $(Price_{Today} - Price_{-4}) / Price_{-4}$
* **NaN Handling:** Keeps the first 4 rows (which contain NaNs)

In [ ]:
for p in passes:

    # --- TRAIN ---
    X_train_df = p["train_df"].copy()
    X_train_df['Return'] = (X_train_df['Close'].shift(-4) - X_train_df['Close']) / X_train_df['Close']
    X_train_df['Return'] = X_train_df['Return'].shift(4)

    # DROP Last 5 rows
    X_train_df = X_train_df.iloc[:-5]

    # Save
    p["X_train_df"] = X_train_df

    # --- VALIDATION ---
    X_val_df = p["val_df"].copy()
    X_val_df['Return'] = (X_val_df['Close'].shift(-4) - X_val_df['Close']) / X_val_df['Close']
    X_val_df['Return'] = X_val_df['Return'].shift(4)

    # DROP Last 5 rows
    X_val_df = X_val_df.iloc[:-5]

    # Save
    p["X_val_df"] = X_val_df


    # --- TEST ---
    X_test_df = p["test_df"].copy()
    X_test_df['Return'] = (X_test_df['Close'].shift(-4) - X_test_df['Close']) / X_test_df['Close']
    X_test_df['Return'] = X_test_df['Return'].shift(4)

    # DROP Last 5 rows
    X_test_df = X_test_df.iloc[:-5]

    # Save
    p["X_test_df"] = X_test_df


In [ ]:
X_train_p = {}
X_val_p = {}
X_test_p = {}

for i in range(len(passes)):
    X_train_p[i+1] = passes[i]["X_train_df"]
    X_val_p[i+1]   = passes[i]["X_val_df"]
    X_test_p[i+1]  = passes[i]["X_test_df"]

In [ ]:
X_train_p

In [ ]:
save_dir = "X_splits"
os.makedirs(save_dir, exist_ok=True)

for p in range(1, len(passes)+1):
    X_train_p[p].to_csv(os.path.join(save_dir, f"pass{p}_train.csv"))
    X_val_p[p].to_csv(os.path.join(save_dir, f"pass{p}_val.csv"))
    X_test_p[p].to_csv(os.path.join(save_dir, f"pass{p}_test.csv"))

print("✅ Saved all X_train_p, X_val_p, X_test_p as CSV.")

## Visual

In [ ]:
X_train_p[1]

In [ ]:
X_train_p[10].tail(11)

In [ ]:
import numpy as np
import pandas as pd

window_size = 5
feature_cols = ['Close','Volume','High','Low','Open','MA20','Close_Original','Return']

def build_tensor_from_df(df, window_size=5, feature_cols=None):
    if feature_cols is None:
        feature_cols = ['Close','Volume','High','Low','Open','MA20','Close_Original','Return']
    
    # 1) Work on a copy
    df_proc = df.copy()

    # 2) Insert duplicate of Close right after MA20 → Close_Original
    if "MA20" in df_proc.columns and "Close" in df_proc.columns and "Close_Original" not in df_proc.columns:
        df_proc.insert(
            df_proc.columns.get_loc("MA20") + 1,
            "Close_Original",
            df_proc["Close"]
        )

    # 3) Keep only the features we care about (and that exist)
    cols = [c for c in feature_cols if c in df_proc.columns]
    if not cols:
        return {
            "X_image": np.empty((0, window_size, 0), dtype=np.float32),
            "cols": [],
            "window_size": window_size,
            "rows_used": 0,
            "num_samples": 0
        }

    df_proc = df_proc[cols]          # you can add .bfill().ffill() if needed
    F = df_proc.to_numpy(dtype=np.float32)  # (N, Fdim)
    N, Fdim = F.shape

    # 4) number of samples: you used N - window_size + 1
    S = N - window_size +1
    if S <= 0:
        return {
            "X_image": np.empty((0, window_size, Fdim), dtype=np.float32),
            "cols": cols,
            "window_size": window_size,
            "rows_used": N,
            "num_samples": 0
        }

    # 5) Sliding windows (S, window_size, Fdim)
    X = np.stack([F[i:i+window_size] for i in range(S)], axis=0)

    # 6) Overwrite 'Close' with compounded path, keep Close_Original
    if 'Close' in cols:
        price_idx = cols.index('Close')
        close = X[:, :, price_idx]  # (S, window_size)

        r_step = np.zeros_like(close)
        r_step[:, 1:] = (close[:, 1:] / close[:, :-1]) - 1.0

        P = np.ones_like(close)
        P[:, 1:] = np.cumprod(1.0 + r_step[:, 1:], axis=1)

        X[:, :, price_idx] = P

    # 7) Transform columns 2,3,4,5 using Close_Original (col 6) and Close (col 0)
    #    (follow your original logic, but safer with index lookups)
    if 'Close_Original' in cols:
        idx_close_orig = cols.index('Close_Original')
        idx_close = cols.index('Close') if 'Close' in cols else None

        for k in [2, 3, 4, 5]:  # High, Low, Open, MA20 (according to feature_cols order)
            if k < Fdim and idx_close is not None:
                X[:, :, k] = (X[:, :, k] / X[:, :, idx_close_orig]) * X[:, :, idx_close]

    # 8) Drop Close_Original & Return (indices 6 and 7 in your design)
    col_to_drop = []
    if 'Close_Original' in cols:
        col_to_drop.append(cols.index('Close_Original'))
    if 'Return' in cols:
        col_to_drop.append(cols.index('Return'))

    if col_to_drop:
        X = np.delete(X, col_to_drop, axis=2)
        # also drop names from cols list
        cols_clean = [c for i, c in enumerate(cols) if i not in col_to_drop]
    else:
        cols_clean = cols

    return {
        "X_image": X,
        "cols": cols_clean,
        "window_size": window_size,
        "rows_used": N,
        "num_samples": S,
    }

In [ ]:
X_train_img_p = {}
X_val_img_p   = {}
X_test_img_p  = {}

for k in range(1, len(passes)+1):
    print(f"\n📊 Building tensors for PASS {k}...")

    # TRAIN
    train_res = build_tensor_from_df(X_train_p[k], window_size=5, feature_cols=feature_cols)
    X_train_img_p[k] = train_res["X_image"]
    print(f"  ✅ Train: shape = {train_res['X_image'].shape}, features = {len(train_res['cols'])}")

    # VAL
    val_res = build_tensor_from_df(X_val_p[k], window_size=5, feature_cols=feature_cols)
    X_val_img_p[k] = val_res["X_image"]
    print(f"  ✅ Val:   shape = {val_res['X_image'].shape}, features = {len(val_res['cols'])}")

    # TEST
    test_res = build_tensor_from_df(X_test_p[k], window_size=5, feature_cols=feature_cols)
    X_test_img_p[k] = test_res["X_image"]
    print(f"  ✅ Test:  shape = {test_res['X_image'].shape}, features = {len(test_res['cols'])}")

In [ ]:
df = X_train_p[1]     # choose any pass

result = build_tensor_from_df(df)

S = result["num_samples"]
N = result["rows_used"]
window_size = result["window_size"]

# ---- FIRST WINDOW ----
start_idx_first = 0
end_idx_first   = window_size - 1

first_window_start_date = df.index[start_idx_first]
first_window_end_date   = df.index[end_idx_first]

print("FIRST window uses rows :", start_idx_first, "→", end_idx_first)
print("FIRST window start date:", first_window_start_date)
print("FIRST window end date  :", first_window_end_date)

# ---- LAST WINDOW (your previous version) ----
start_idx_last = S - 1
end_idx_last   = start_idx_last + window_size - 1

last_window_start_date = df.index[start_idx_last]
last_window_end_date   = df.index[end_idx_last]

print("\nLAST window uses rows  :", start_idx_last, "→", end_idx_last)
print("LAST window start date :", last_window_start_date)
print("LAST window end date   :", last_window_end_date)

In [ ]:
X_train_p[1].tail(11)

In [ ]:
# ----------------------------- helpers -----------------------------
def _normalize(arr, eps=1e-9):
    arr = np.asarray(arr, dtype=float)
    lo, hi = float(arr.min()), float(arr.max())
    if hi - lo < eps:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# -------------------- main renderer (grayscale) --------------------
def ohlc_with_ma_volume(
    open_, high, low, close,
    volume=None, ma=None,
    px_per_bar=3,
    margin_px=2,
    H=32,
    price_to_vol_ratio=(4,1),   # top:bottom split
    wick_w=1,
    tick_w=1,
    tick_len=1,                 # open/close tick length
    extend_ma_frac=0.2,
    fg=255, bg=0
):
    """
    Minimal OHLC bars (wick center; open tick left; close tick right) + MA + slim volume.
    Draws on an (H, W) canvas, then crops 2 px from left and right -> final shape (64, 60).
    """
    open_, high, low, close = map(lambda a: np.asarray(a, float), (open_, high, low, close))
    n = len(close)
    assert len(open_) == len(high) == len(low) == n

    # Layout (native draw width)
    W = n * px_per_bar + 2 * margin_px
    if volume is not None:
        p, v = price_to_vol_ratio
        price_H = int(round(H * (p / (p + v))))
        vol_H   = H - price_H
    else:
        price_H = H
        vol_H   = 0
    y0_price, y0_vol = 0, price_H

    # Canvas
    img = Image.new("L", (W, H), color=bg)
    draw = ImageDraw.Draw(img)

    # Y-scale for price
    lo_min = float(low.min()); hi_max = float(high.max())
    pad = (hi_max - lo_min) * 0.03
    lo_min -= pad; hi_max += pad
    rng = max(hi_max - lo_min, 1e-9)

    def ypix(v):
        t = (v - lo_min) / rng
        t = 0.0 if t < 0 else 1.0 if t > 1 else t
        return y0_price + (price_H - 1) - int(round(t * (price_H - 1)))

    # X centers
    cx = np.arange(n) * px_per_bar + margin_px + px_per_bar // 2

    # OHLC bars (wick + ticks)
    for i in range(n):
        x = int(cx[i])
        yH, yL = ypix(high[i]), ypix(low[i])
        yO, yC = ypix(open_[i]), ypix(close[i])
        draw.line([(x, yH), (x, yL)], fill=fg, width=wick_w)                 # wick
        draw.line([(x - tick_len, yO), (x, yO)], fill=fg, width=tick_w)      # open (left)
        draw.line([(x, yC), (x + tick_len, yC)], fill=fg, width=tick_w)      # close (right)

    # MA line (optional)
    if ma is not None:
        ma = np.asarray(ma, float)
        pts = [(int(cx[i]), ypix(ma[i])) for i in range(n)]
        dx = px_per_bar * extend_ma_frac
        left_ext  = (int(round(cx[0] - dx+1)), ypix(ma[0]))
        right_ext = (int(round(cx[-1] + dx-1)), ypix(ma[-1]))
        pts = [left_ext] + pts + [right_ext]
        for a, b in zip(pts[:-1], pts[1:]):
            draw.line([a, b], fill=fg, width=1)

    # Volume: slim centered line per day
    if volume is not None and vol_H > 0:
        vol = np.asarray(volume, float)
        norm = _normalize(vol)
        for i in range(n):
            x = int(cx[i])
            h = int(round(norm[i] * (vol_H - 1)))
            h = max(h, 1)
            y_bottom = y0_vol + vol_H - 1
            y_top    = y_bottom - h
            draw.line([(x, y_bottom), (x, y_top)], fill=fg, width=1)

    # --- return as array, then crop 2 px from left & right -> (64, 60) ---
    arr = np.array(img, dtype=np.uint8)
    # # only crop if width >= 64; remove exactly 2 px on both sides
    if arr.shape[1] >= 32:
        arr = arr[:, 2:-2]  # (H, W-4) -> if W was 64, becomes 60
    # If somehow wider than 64 originally, center-crop to 60 for safety:
    if arr.shape[1] > 15:
        start = (arr.shape[1] - 15) // 2
        arr = arr[:, start:start + 15]
    return arr  # final (32, 15)

# ------------------------ small batch + preview ------------------------
def make_batch(commodity_features, n_samples=10):
    imgs = []
    for name, tensors in commodity_features.items():
        X_image_train = tensors["X_image_train"]
        for idx in range(len(X_image_train)):
            close, vol, high, low, open_, ma = X_image_train[idx,:,0], X_image_train[idx,:,1], X_image_train[idx,:,2], X_image_train[idx,:,3], X_image_train[idx,:,4], X_image_train[idx,:,5]
            arr = ohlc_with_ma_volume(open_, high, low, close, volume=vol, ma=ma, H=32, px_per_bar=3, margin_px=2)
            imgs.append(arr)
            if len(imgs) >= n_samples:
                return imgs
    return imgs

def preview_grid(imgs, ncols=5, scale=6):
    if not imgs:
        print("No images to preview."); return
    n = len(imgs)
    nrows = (n + ncols - 1) // ncols
    plt.figure(figsize=(ncols * scale, nrows * scale))
    for i, arr in enumerate(imgs):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(arr, cmap="gray", vmin=0, vmax=255)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

def preview_grid(imgs, ncols=5, scale=4):
    """
    Show images in a grid with bigger display size.
    scale controls how large each image cell looks.
    """
    if not imgs:
        print("No images to preview."); return
    n = len(imgs)
    nrows = (n + ncols - 1) // ncols
    plt.figure(figsize=(ncols * scale, nrows * scale))
    for i, arr in enumerate(imgs):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(arr, cmap="gray", vmin=0, vmax=255)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Take one window from pass 10
w = X_train_img_p[1][18]          # shape (5, 6)

# Unpack columns: [Close, Volume, High, Low, Open, MA20]
close = w[:, 0]
vol   = w[:, 1]
high  = w[:, 2]
low   = w[:, 3]
open_ = w[:, 4]
ma    = w[:, 5]

# Use your renderer to make ONE image
img = ohlc_with_ma_volume(
    open_, high, low, close,
    volume=vol,
    ma=ma,
    H=32,
    px_per_bar=3,
    margin_px=2
)

# Show it
plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.show()

In [ ]:
N_frames = 50
frames = []

SCALE = 12
# 60 is a good size for the text when the image is large
FONT_SIZE = 25

# --- FONT LOADING ---
font = None
possible_fonts = [
    "arial.ttf", 
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
]

for path in possible_fonts:
    try:
        font = ImageFont.truetype(path, FONT_SIZE)
        break
    except (OSError, IOError):
        continue

if font is None:
    font = ImageFont.load_default() # Fallback (will be small)
# --------------------

for idx in range(N_frames):
    w = X_train_img_p[1][idx]

    close = w[:, 0]
    vol   = w[:, 1]
    high  = w[:, 2]
    low   = w[:, 3]
    open_ = w[:, 4]
    ma    = w[:, 5]

    # 1. Render raw (small) image
    img_small = ohlc_with_ma_volume(
        open_, high, low, close,
        volume=vol, ma=ma,
        H=32, px_per_bar=3, margin_px=2
    )
    
    img_pil = Image.fromarray(img_small)

    # 2. Upscale FIRST (Make it Big)
    W_big = img_small.shape[1] * SCALE
    H_big = img_small.shape[0] * SCALE
    img_big = img_pil.resize((W_big, H_big), resample=Image.NEAREST)

    # 3. Draw Text on the BIG Image (Clean & Sharp)
    if img_big.mode != 'RGB':
        img_big = img_big.convert('RGB')

    draw = ImageDraw.Draw(img_big)
    text = str(idx + 1)

    # Calculate position (Top Right corner, slightly padded)
    # Using textbbox to center or position accurately
    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    
    # Position: Top Right (Width - TextWidth - Padding, TopPadding)
    x_pos = W_big - text_width - 20 
    y_pos = 20

    draw.text((x_pos, y_pos), text, font=font, fill=(255, 255, 255))

    frames.append(np.array(img_big))

# Save
gif_path = "pass1_animation_sharp_text.gif"
imageio.mimsave(gif_path, frames, duration=0.3)
print("GIF saved as:", gif_path)

# Display
plt.figure(figsize=(6, 6))
plt.imshow(frames[0])
plt.axis("off")
plt.title("Sharp Text on Pixel Chart")
plt.show()

In [ ]:
# Base folder for all images
base_dir = "charts_passes"
os.makedirs(base_dir, exist_ok=True)

num_passes = len(passes)

for k in range(1, num_passes + 1):  # pass 1..10
    
    print(f"\n📊 PASS {k}: Processing train/val/test...")

    # --------------- LOOP OVER SPLITS -----------------
    for split_name, X_split in [
        ("train", X_train_img_p[k]),
        ("val",   X_val_img_p[k]),
        ("test",  X_test_img_p[k])
    ]:

        S = X_split.shape[0]

        # Make directory for this pass and split
        out_dir = os.path.join(base_dir, f"pass{k}", split_name)
        os.makedirs(out_dir, exist_ok=True)

        print(f"   ➤ {split_name} : {S} windows")

        for idx in range(S):
            w = X_split[idx]  # shape (5, 6)

            # Unpack OHLCVMA
            close = w[:, 0]
            vol   = w[:, 1]
            high  = w[:, 2]
            low   = w[:, 3]
            open_ = w[:, 4]
            ma    = w[:, 5]

            # Render OHLC+MA+Volume chart
            img_arr = ohlc_with_ma_volume(
                open_, high, low, close,
                volume=vol,
                ma=ma,
                H=32,
                px_per_bar=3,
                margin_px=2
            )

            # Save PNG
            img_pil = Image.fromarray(img_arr)
            fname = f"pass{k}_{split_name}_{idx:05d}.png"
            fpath = os.path.join(out_dir, fname)
            img_pil.save(fpath)

        print(f"   ✅ Saved {S} images to {out_dir}")

print("\n🎉 ALL PASSES COMPLETED — IMAGES SAVED IN 'charts_passes/'")

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

base_dir = "charts_passes"

# storage: image_df[pass_id]["train"][i] -> DataFrame
image_df = {}

for k in range(1, len(passes)+1):  # pass1..pass10
    print(f"\n📥 Loading PASS {k} images...")
    
    image_df[k] = {"train": [], "val": [], "test": []}

    for split in ["train", "val", "test"]:
        folder = os.path.join(base_dir, f"pass{k}", split)
        
        if not os.path.exists(folder):
            print(f"❌ Folder missing: {folder}")
            continue
        
        files = sorted([f for f in os.listdir(folder) if f.endswith(".png")])
        print(f"   ➤ {split}: {len(files)} images found")

        for fname in files:
            path = os.path.join(folder, fname)

            # Load image → grayscale array
            arr = np.array(Image.open(path).convert("L"))

            # Convert to DataFrame
            df_img = pd.DataFrame(arr)

            image_df[k][split].append(df_img)

print("\n✅ COMPLETED: All images loaded back as pandas DataFrames!")